In [4]:
import openai
import pprint

In [5]:
client = openai.OpenAI(base_url="http://localhost:11434/v1",api_key="asdf")

In [6]:
import random
import json


def handle_tools_call(message):
    response = []
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_names":
        arguments = json.loads(tool_call.function.arguments)
        start_name = arguments.get("start_name")
        name = get_names()
        response.append({
            "role": "tool",
            "content": name,
            "tool_call_id": tool_call.id
        })
    return response


def get_names():
    names = ["simran", "raj", "pragathi", "chethan", "swapnil", "luck"]
    return "raj"


name_function = {
    "name": "get_names",
    "description": "Get a random name from the list",
    "parameters": {
        "type": "object",
        "properties": {
            "start_name": {
                "type": "string",
                "description": "Starting letter with which user wants the name to start with",
            }
        },
        "required": ["start_name"],
        "additionalProperties": False
    }
}


tools = [{"type": "function", "function": name_function}]

In [9]:
input = [{"role":"user","content":"= a name starting with r"}]
response = client.chat.completions.create(model="llama3.2",messages=input,tools=tools)

print(response)    


ChatCompletion(id='chatcmpl-366', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_uqo7jshz', function=Function(arguments='{"start_name":"r"}', name='get_names'), type='function', index=0)]))], created=1776249796, model='llama3.2', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=18, prompt_tokens=165, total_tokens=183, completion_tokens_details=None, prompt_tokens_details=None))


In [10]:
while response.choices[0].finish_reason == "tool_calls":
    message = response.choices[0].message
    tool_response = handle_tools_call(message)
    input.append(message)
    input.extend(tool_response)
    response = client.chat.completions.create(model="llama3.2",messages=input,tools=tools)
print(response.choices[0].message)

ChatCompletionMessage(content='The first result for names starting with R is: raj.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [ ]:
ChatCompletion(id='chatcmpl-660', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, 

message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, 
tool_calls=[
    ChatCompletionMessageFunctionToolCall(id='call_n1w3gduv', function=Function(arguments='{"start_name":["r"]}', name='get_names'), type='function', index=0)]
    ))], created=1776246048, model='llama3.2', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=19, prompt_tokens=165, total_tokens=184, completion_tokens_details=None, prompt_tokens_details=None))
